# 01 · Build source-space graphs (pipeline v2)

Thin driver: all logic lives in `src/adgraph`. Parameters: `configs/pipeline.yaml`. Outputs go to a new timestamped folder in `AD-LEB/processed/` together with a manifest. Nothing is overwritten.

Run the cells in order. **Stop after step 3** if the header report shows missing channels or unexpected sampling rates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Code and environment

In [ ]:
BRANCH = 'claude/ecstatic-davinci-a2e8mh'  # change to main once merged
!rm -rf /content/repo && git clone -q -b $BRANCH https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .
!git rev-parse HEAD

## 2. Paths
`configs/paths.yaml` is not committed; it is created here from the template (edit `drive_root` if your shortcut differs).

In [ ]:
!cp -n configs/paths.example.yaml configs/paths.yaml && cat configs/paths.yaml

## 3. Header check (reads no signal)
Expect 20 D1 + 23 D2 rows, empty `missing_canonical`. Anything listed in `unrecognised` is dropped (e.g. ECG/EOG).

In [ ]:
!python scripts/inspect_raw.py --paths configs/paths.yaml --out /content/drive/MyDrive/AD-LEB/processed

## 4. Smoke test: first subject of each dataset
Also downloads fsaverage (~200 MB) into the Drive cache once, and verifies the shared sensor→ROI operator against the full MNE inverse.

In [ ]:
!python scripts/build_graphs.py --paths configs/paths.yaml --limit 1 --out-root /content/smoke

## 5. Full run

In [ ]:
!python scripts/build_graphs.py --paths configs/paths.yaml

## 6. QC summary

In [ ]:
import glob, json, pandas as pd
out = sorted(glob.glob('/content/drive/MyDrive/AD-LEB/processed/graphs_v2_*'))[-1]
m = json.load(open(f'{out}/manifest.json'))
print(out, '\ncommit', m['git_commit'], '| config', m['config_hash'], '| operator err', m['source_operator']['verification_rel_error'])
print(json.dumps(m['summary']['per_dataset'], indent=1), m['summary']['subjects_not_ok'])
s = pd.read_csv(f'{out}/subjects.csv')
s[['key','label','orig_sfreq','duration_s','median_std_uv','n_bad_channels','n_windows_kept','n_windows_total','status']]